# exp013 NB2 v2 — R1 v2: HGNetV2-B0 + Tucker recipe + Perch distill + pseudo

Tucker recipe を **HGNetV2-B0 backbone** に変更し、arch diversity で R1 v1 失敗 (Tucker クローン化) を回避。
**Drive-based 構成**: Tucker 97GB zip 廃止、train_audio/SS は Drive から copy、.ogg 直読。

## なぜ HGNetV2-B0?
- Tucker は EffNet-B0 SED、我々の v8 にも入ってる → 同 arch だと blend で重複
- **HGNetV2-B0** は別 arch (GhostConv 系)、Tucker と独立な特徴抽出経路
- **Natsume が BC2026 Discussion (5/10) で実証**: hgnetv2_b0 solo 0.931 → +pseudo concat で **0.943 (+0.012)** 達成
- BC2025 12位 223223 も HGNetV2 採用

## データソース (Drive 経由、97GB DL 廃止)
| Source | サイズ | 取得方法 | ラベル | Share |
|---|---|---|---|---|
| **train_audio** (.ogg) | ~15 GB | Drive copy ~10-15 min | focal=hard | 0.85 |
| **train_soundscapes** (.ogg) | ~5 GB | Drive copy ~3-5 min | sc=hard / pseudo soft | 0.05 / 0.10 |
| Tucker meta CSVs | 2.5 MB | Drive copy (audio_cache_meta + sc_cache_meta) | - | - |
| Pseudo CSV (NB1 v2 出力) | 370 MB | Kaggle DL ~1 min | soft | (sc に統合) |
| Perch ONNX + wheel | 1.5 GB | Kaggle DL ~3 min | distillation teacher | - |
| **合計 disk** | **~22 GB** (vs 旧 112 GB) | | | |

## 学習
- Backbone: **hgnetv2_b0.ssld_stage2_ft_in1k** (SSLD pretraining)
- Distillation: ON (alpha=1.0, MSE to Perch v2 1536-d、Tucker 流の stop-grad 構造)
- Loss: 0.5 * BCE_clip + 0.5 * BCE_frame_max + 1.0 * MSE_distill
- **MIXUP_HARD = False** (soft label を保持、pseudo の soft 性に整合)
- **LR = 3e-4** (HGNet は 5e-4 不安定報告)
- Folds: **[0] のみ** (AUC 確認後 [1..4] 追加)
- 25 epoch, batch=64 (A100 40GB)

## 出力
- `/content/output/r1v2_fold{k}_best_ns22.pth`
- `/content/output/r1v2_fold{k}_best_macro.pth`
- `/content/output/r1v2_fold{k}.onnx`
- すべて `/content/drive/MyDrive/birdclef2026/exp013/r1v2/` にミラー

## 想定時間 (Colab Pro A100, 1 fold)
- データ DL: 30-50 min (zip 97GB が支配的)
- 学習 25 epoch: 1.5-2.5h
- ONNX export + Drive mirror: 5 min
- **合計 1 fold: ~2-3.5h**


In [2]:
# ============================================================
# Cell 1: Setup — Drive mount, project root, kaggle.json
# ============================================================
!pip install -q timm onnx onnxruntime-gpu librosa kaggle scipy soundfile

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, json, shutil, time, subprocess
from pathlib import Path

# ★ ユーザー指定の Drive folder (memory: user_colab_drive_folder.md)
DRIVE_INPUT_DIR = Path("/content/drive/MyDrive/kaggle/birdclef2026")
assert DRIVE_INPUT_DIR.exists(), f"Drive input folder missing: {DRIVE_INPUT_DIR}"

DRIVE_OUTPUT_DIR = DRIVE_INPUT_DIR / "output" / "exp013" / "r1v2"
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Drive input:  {DRIVE_INPUT_DIR}")
print(f"Drive output: {DRIVE_OUTPUT_DIR}")

# Search for project code (experiment/exp013) on Drive — may be different from input data dir
print("\nSearching for project code on Drive...")
r = subprocess.run(
    ["find", "/content/drive/MyDrive", "-maxdepth", "6", "-name", "birdclef-2026", "-type", "d"],
    capture_output=True, text=True)
candidates = [Path(p) for p in r.stdout.strip().split("\n") if p]
PROJECT_ROOT = None
for c in candidates:
    if (c / "experiment" / "exp013").exists():
        PROJECT_ROOT = c; break
if PROJECT_ROOT is None and candidates:
    PROJECT_ROOT = candidates[0]
print(f"PROJECT_ROOT (code): {PROJECT_ROOT}")

# kaggle.json setup
KJ_CANDIDATES = [
    Path("/content/drive/MyDrive/kaggle.json"),
    DRIVE_INPUT_DIR / "kaggle.json",
]
if PROJECT_ROOT is not None:
    KJ_CANDIDATES.append(PROJECT_ROOT / "kaggle.json")
KJ = next((p for p in KJ_CANDIDATES if p.exists()), None)
assert KJ is not None, f"kaggle.json not found in {KJ_CANDIDATES}"
KAGGLE_CFG = Path.home() / ".kaggle"
KAGGLE_CFG.mkdir(parents=True, exist_ok=True)
shutil.copy(str(KJ), str(KAGGLE_CFG / "kaggle.json"))
os.chmod(str(KAGGLE_CFG / "kaggle.json"), 0o600)
creds = json.loads(KJ.read_text())
if creds.get("key", "").startswith("KGAT_"):
    os.environ["KAGGLE_API_TOKEN"] = creds["key"]
print(f"kaggle.json: {KJ}")

# Working dirs
OUT_DIR_DRIVE = DRIVE_OUTPUT_DIR   # alias for backward compat in later cells

LOCAL_DATA = Path("/content/data")
LOCAL_OUT  = Path("/content/output")
LOCAL_DATA.mkdir(parents=True, exist_ok=True)
LOCAL_OUT.mkdir(parents=True, exist_ok=True)

print(f"\nDrive output: {OUT_DIR_DRIVE}")
print(f"Local data:   {LOCAL_DATA}")
print(f"Local output: {LOCAL_OUT}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive input:  /content/drive/MyDrive/kaggle/birdclef2026
Drive output: /content/drive/MyDrive/kaggle/birdclef2026/output/exp013/r1v2

Searching for project code on Drive...
PROJECT_ROOT (code): /content/drive/MyDrive/birdclef-2026
kaggle.json: /content/drive/MyDrive/kaggle/birdclef2026/kaggle.json

Drive output: /content/drive/MyDrive/kaggle/birdclef2026/output/exp013/r1v2
Local data:   /content/data
Local output: /content/output


In [3]:
# ============================================================
# Cell 2: Data prep — Drive copy + Kaggle download (NO Tucker waveform-cache)
# ============================================================
# v2 (Drive-based): Tucker 97GB zip 廃止、train_audio/SS は Drive から copy、.ogg 直読
import time, zipfile, shutil, subprocess
from pathlib import Path
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi(); api.authenticate()
print("kaggle authenticated")

T0_total = time.time()

# 1) Competition CSVs (Drive から copy — DRIVE_INPUT_DIR 優先、無ければ PROJECT_ROOT)
comp = LOCAL_DATA / "competition"
comp.mkdir(parents=True, exist_ok=True)
for fn in ["train.csv", "taxonomy.csv", "sample_submission.csv", "train_soundscapes_labels.csv"]:
    src_candidates = [DRIVE_INPUT_DIR / fn]
    if PROJECT_ROOT is not None:
        src_candidates.append(PROJECT_ROOT / fn)
    src = next((s for s in src_candidates if s.exists()), None)
    dst = comp / fn
    if src is not None and not dst.exists():
        shutil.copy2(str(src), str(dst))
        print(f"  competition/{fn} OK (from Drive: {src.parent.name})")
    elif src is None:
        # Fallback: download from Kaggle
        print(f"  {fn} not on Drive, DL from Kaggle...")
        api.competition_download_file("birdclef-2026", file_name=fn, path=str(comp))
        z = comp / (fn + ".zip")
        if z.exists():
            with zipfile.ZipFile(z) as zf:
                zf.extractall(comp)
            z.unlink()

# 2) Tucker meta CSVs (Drive にあれば copy、なければ Kaggle DL)
#    waveform-cache の zip 自体は不要、CSV だけ欲しい
WC_DIR = LOCAL_DATA / "waveform-cache"
(WC_DIR / "waveform_cache").mkdir(parents=True, exist_ok=True)
META_NEEDED = ["audio_cache_meta.csv", "soundscape_cache_meta.csv", "soundscape_file_meta.csv"]
# Try Drive paths
DRIVE_META_CANDIDATES = [
    DRIVE_INPUT_DIR / "tucker_meta",
]
if PROJECT_ROOT is not None:
    DRIVE_META_CANDIDATES += [
        PROJECT_ROOT / "experiment" / "exp013" / "notebook" / "tucker_meta_test",
        PROJECT_ROOT / "tucker_meta",
    ]
for fn in META_NEEDED:
    dst = WC_DIR / "waveform_cache" / fn
    if dst.exists():
        print(f"  meta {fn} already present")
        continue
    src_found = None
    for cand_dir in DRIVE_META_CANDIDATES:
        src = cand_dir / fn
        if src.exists():
            src_found = src
            break
    if src_found:
        shutil.copy2(str(src_found), str(dst))
        print(f"  meta {fn} OK (from Drive)")
    else:
        # Fallback: download from Tucker dataset
        print(f"  meta {fn} not on Drive, DL from Kaggle...")
        try:
            api.dataset_download_file(
                "tuckerarrants/birdclef-2026-waveform-cache",
                file_name=f"waveform_cache/{fn}",
                path=str(WC_DIR / "waveform_cache"),
            )
            z = WC_DIR / "waveform_cache" / (fn + ".zip")
            if z.exists():
                with zipfile.ZipFile(z) as zf:
                    zf.extractall(WC_DIR / "waveform_cache")
                z.unlink()
        except Exception as e:
            print(f"    failed: {e}")
            if fn != "soundscape_file_meta.csv":
                raise  # required
            print(f"    {fn} optional, will disable focal-SC mixup")

# 3) train_audio (Drive → /content/data)
TA_DIR = LOCAL_DATA / "train_audio"
DRIVE_TA = DRIVE_INPUT_DIR / "train_audio"
if not TA_DIR.exists() or sum(1 for _ in TA_DIR.rglob("*.ogg")) < 1000:
    if DRIVE_TA.exists():
        n_drive = sum(1 for _ in DRIVE_TA.rglob("*.ogg"))
        print(f"\nDrive train_audio: {n_drive} .ogg files at {DRIVE_TA}")
        print(f"Copying to local /content/data/train_audio (~15GB, ~10-15 min)...")
        t0 = time.time()
        shutil.copytree(str(DRIVE_TA), str(TA_DIR), dirs_exist_ok=True)
        n = sum(1 for _ in TA_DIR.rglob("*.ogg"))
        print(f"  done in {time.time()-t0:.0f}s, {n} .ogg files (drive had {n_drive})")
    else:
        raise FileNotFoundError(
            f"train_audio not on Drive at {DRIVE_TA}. "
            f"Drive アップロード未完了 or path 違いの可能性")
else:
    n = sum(1 for _ in TA_DIR.rglob("*.ogg"))
    print(f"\ntrain_audio already present in /content/data: {n} .ogg files")

# 4) train_soundscapes (Drive → /content/data)
TS_DIR = LOCAL_DATA / "train_soundscapes"
DRIVE_TS = DRIVE_INPUT_DIR / "train_soundscapes"
if not TS_DIR.exists() or sum(1 for _ in TS_DIR.glob("*.ogg")) < 1000:
    if DRIVE_TS.exists():
        n_drive = sum(1 for _ in DRIVE_TS.glob("*.ogg"))
        print(f"\nDrive train_soundscapes: {n_drive} .ogg files at {DRIVE_TS}")
        print(f"Copying to local (~5GB, ~3-5 min)...")
        t0 = time.time()
        shutil.copytree(str(DRIVE_TS), str(TS_DIR), dirs_exist_ok=True)
        n_ogg = sum(1 for _ in TS_DIR.glob("*.ogg"))
        print(f"  done in {time.time()-t0:.0f}s, {n_ogg} .ogg files (drive had {n_drive})")
    else:
        raise FileNotFoundError(
            f"train_soundscapes not on Drive at {DRIVE_TS}. "
            f"Drive アップロード未完了 or path 違いの可能性")
else:
    n_ogg = sum(1 for _ in TS_DIR.glob("*.ogg"))
    print(f"\ntrain_soundscapes already present in /content/data: {n_ogg} .ogg files")

# 5) Pseudo CSV (NB1 v2 出力)
PSEUDO_DIR = LOCAL_DATA / "pseudo-r1-v2"
PSEUDO_CSV = PSEUDO_DIR / "pseudo_labels.csv"
if not PSEUDO_CSV.exists():
    PSEUDO_DIR.mkdir(parents=True, exist_ok=True)
    print(f"\nDownloading pseudo-r1-v2 CSV...")
    t0 = time.time()
    api.dataset_download_files(
        "maekeso/birdclef2026-exp013-pseudo-r1-v2",
        path=str(PSEUDO_DIR), unzip=True, quiet=False)
    # Dataset may unzip to a nested folder; locate the CSV
    if not PSEUDO_CSV.exists():
        cands = list(PSEUDO_DIR.rglob("pseudo_labels.csv"))
        if cands:
            shutil.move(str(cands[0]), str(PSEUDO_CSV))
    print(f"  done in {time.time()-t0:.0f}s, {PSEUDO_CSV.stat().st_size/1e6:.1f}MB")
else:
    print(f"\npseudo CSV already present: {PSEUDO_CSV.stat().st_size/1e6:.1f}MB")

# 6) Perch ONNX (1.5GB)
po_dir = LOCAL_DATA / "perch-onnx"
if not (po_dir / "perch_v2_no_dft.onnx").exists():
    po_dir.mkdir(parents=True, exist_ok=True)
    print("\nDownloading perch-onnx...")
    t0 = time.time()
    api.dataset_download_files(
        "tuckerarrants/perch-v2-no-dft-onnx",
        path=str(po_dir), unzip=True, quiet=False)
    print(f"  done in {time.time()-t0:.0f}s")
else:
    print("\nperch-onnx already present")

# 7) Disk summary
print(f"\n=== Total DL time: {(time.time()-T0_total)/60:.1f} min ===")
print("\n=== /content disk ===")
r = subprocess.run(["df", "-h", "/content"], capture_output=True, text=True)
print(r.stdout)

print("=== /content/data summary ===")
for sub in sorted(LOCAL_DATA.iterdir()):
    if sub.is_dir():
        n = sum(1 for _ in sub.rglob("*") if _.is_file())
        sz = sum(_.stat().st_size for _ in sub.rglob("*") if _.is_file()) / 1e9
        print(f"  {sub.name}/  ({n} files, {sz:.1f}GB)")


kaggle authenticated
  competition/train.csv OK (from Drive: birdclef2026)
  competition/taxonomy.csv OK (from Drive: birdclef2026)
  competition/sample_submission.csv OK (from Drive: birdclef2026)
  competition/train_soundscapes_labels.csv OK (from Drive: birdclef2026)
  meta audio_cache_meta.csv not on Drive, DL from Kaggle...
Dataset URL: https://www.kaggle.com/datasets/tuckerarrants/birdclef-2026-waveform-cache
  meta soundscape_cache_meta.csv not on Drive, DL from Kaggle...
Dataset URL: https://www.kaggle.com/datasets/tuckerarrants/birdclef-2026-waveform-cache
  meta soundscape_file_meta.csv not on Drive, DL from Kaggle...
Dataset URL: https://www.kaggle.com/datasets/tuckerarrants/birdclef-2026-waveform-cache

Drive train_audio: 35549 .ogg files at /content/drive/MyDrive/kaggle/birdclef2026/train_audio
Copying to local /content/data/train_audio (~15GB, ~10-15 min)...


KeyboardInterrupt: 

In [ ]:
# ============================================================
# Cell 3: Install Tucker's bundled onnxruntime wheel (matches Perch ONNX format)
# ============================================================
WHEEL = next(LOCAL_DATA.rglob("onnxruntime-*.whl"), None)
if WHEEL is not None:
    !pip install -q {WHEEL}
    print(f"Installed {WHEEL.name}")
else:
    print("No bundled wheel; using onnxruntime-gpu installed earlier")


In [ ]:
# =================================================================
# S1 -- IMPORTS + CONFIG
# =================================================================
import os, sys, time, json, pickle, gc, random, math
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torch.cuda.amp import GradScaler, autocast
import torchaudio
import timm
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from scipy.special import expit as sigmoid_np
import warnings
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available(): print(f"GPU: {torch.cuda.get_device_name()}")

# ------------------------------------------------------------------
# Notebook Mode
# -----------------------------------------------------------------
MODE = "train"  # "train" or "infer"

DEBUG = False  # Colab Pro: full training

# ------------------------------------------------------------------
# Paths -- Kaggle dataset layout
# -----------------------------------------------------------------
COMP_DIR = Path("/content/data/competition")
WAVEFORM_CACHE_DIR = Path("/content/data/waveform-cache/waveform_cache")  # virtual; actual reads via zip
PERCH_ONNX_PATH = Path("/content/data/perch-onnx/perch_v2_no_dft.onnx")

LABELS_PATH     = COMP_DIR / "train_soundscapes_labels.csv"
TAXONOMY_PATH   = COMP_DIR / "taxonomy.csv"
SAMPLE_SUB_PATH = COMP_DIR / "sample_submission.csv"
TEST_DIR        = COMP_DIR / "test_soundscapes"

OUT_DIR = Path("/content/output")  # mirrored to Drive

NUM_CLASSES = 234
SR = 32000

# --- Duration ---
TRAIN_DURATION = 5    # seconds
VAL_DURATION   = 5    # always 5s for competition eval
TRAIN_SAMPLES  = SR * TRAIN_DURATION
VAL_SAMPLES    = SR * VAL_DURATION

N_FOLDS = 5

# --- Mel spectrogram ---
N_FFT      = 2048
HOP_LENGTH = 512
N_MELS     = 256
FMIN       = 20
FMAX       = 16000

# --- Model ---
BACKBONE_NAME = "hgnetv2_b0.ssld_stage2_ft_in1k"  # ★ Tucker と arch 違い、Natsume 実証 +0.012

# --- Perch distillation ---
USE_PERCH_DISTILL = True
PERCH_EMBED_DIM   = 1536
ALPHA_DISTILL     = 1.0   # MSE loss weight

# --- Training ---
FOLDS  = [0]  # initial 1-fold AUC check; expand to [0,1,2,3,4] after
EPOCHS = 25
BATCH  = 64  # A100 40GB; drop to 32 if V100 OOM        # used 64 locally, 16 for Kaggle
LR     = 3e-4  # ★ HGNetV2 は 5e-4 だと不安定報告あり、保守的に
MIN_LR = 1e-5
WD     = 1e-4
WARMUP_EPOCHS = 2

# --- Upsampling ---
MIN_SAMPLE = 20

# --- Augmentation ---
AUG_PROB = 0.5
AUG_GAIN_DB_RANGE      = (-6.0, 6.0)
AUG_NOISE_SNR_DB_RANGE = (10.0, 30.0)

# --- MixUp ---
USE_FOCAL_MIXUP    = True
MIXUP_PROB         = 0.5
MIXUP_ALPHA        = 0.4
MIXUP_HARD         = False   # ★ soft label を保持 (pseudo の soft 性を活かす)

USE_FOCAL_SC_MIXUP     = True
FOCAL_SC_MIXUP_PROB    = 0.5
FOCAL_SC_MIXUP_ALPHA   = 0.4

# --- FreqMixStyle (disabled by default) ---
FREQ_MIXSTYLE_PROB  = 0.0
FREQ_MIXSTYLE_ALPHA = 0.1

# --- SpecAugment ---
FREQ_MASK_PARAM = 10
TIME_MASK_PARAM = 10
NUM_FREQ_MASKS  = 1
NUM_TIME_MASKS  = 2

# --- Source weights ---
USE_FOCAL           = True
USE_FOCAL_SECONDARY = True
USE_LABELED_SC      = True

# --- v2: pseudo SC source ---
USE_PSEUDO_SC      = True
PSEUDO_SC_SHARE    = 0.20

ACTIVE_SOURCES = ["focal", "sc", "pseudo_sc"]
# ★ Tucker original (focal 0.9, sc 0.1) に近い保守的比率、pseudo は薄く追加
SHARES = {"focal": 0.85, "sc": 0.05, "pseudo_sc": 0.10}
SOURCE_WEIGHTS = {
    "focal":         1.0,
    "focal_missing": 0.0,
    "sc":            1.0,
    "pseudo_sc":     0.5,   # noisier than hard labels
}

# --- pseudo CSV path (NB1 v2 output) ---
PSEUDO_CSV_PATH    = Path("/content/data/pseudo-r1-v2/pseudo_labels.csv")
TRAIN_SC_DIR       = Path("/content/data/train_soundscapes")

print(f"Backbone: {BACKBONE_NAME}")
print(f"Train duration: {TRAIN_DURATION}s | Mel: {N_MELS} mels, n_fft={N_FFT}, hop={HOP_LENGTH}")
print(f"Distillation: {'ON' if USE_PERCH_DISTILL else 'OFF'} (alpha={ALPHA_DISTILL})")
print(f"Batch: {BATCH} | Epochs: {EPOCHS} | Folds: {FOLDS}")


In [ ]:
# =================================================================
# S2 -- LOAD DATA
# =================================================================

# --- Label ordering from sample_submission (defines column order) ---
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
PRIMARY_LABELS = sample_sub.columns[1:].tolist()
LABEL2IDX = {label: idx for idx, label in enumerate(PRIMARY_LABELS)}
taxonomy = pd.read_csv(TAXONOMY_PATH)
label_to_taxon = dict(zip(taxonomy["primary_label"].astype(str),
                          taxonomy["class_name"].astype(str)))
TAXON_MASKS = {t: np.array([i for i, l in enumerate(PRIMARY_LABELS)
                            if label_to_taxon.get(l, "") == t])
               for t in ["Aves", "Amphibia", "Insecta", "Mammalia", "Reptilia"]}

# --- Focal recording metadata ---
audio_cache_meta = pd.read_csv(WAVEFORM_CACHE_DIR / "audio_cache_meta.csv")
train_df = pd.read_csv(COMP_DIR / "train.csv")
audio_cache_meta = audio_cache_meta.merge(
    train_df[["filename", "secondary_labels"]], on="filename", how="left"
)
audio_cache_meta = audio_cache_meta[
    audio_cache_meta["primary_label"].isin(LABEL2IDX)
].reset_index(drop=True)
print(f"Focal audio cache: {len(audio_cache_meta)} entries")

# --- Soundscape window metadata ---
sc_cache_meta = pd.read_csv(WAVEFORM_CACHE_DIR / "soundscape_cache_meta.csv")
sc_cache_meta["label_list"] = sc_cache_meta["label_list"].apply(
    lambda x: x.split(";") if isinstance(x, str) else []
)
print(f"Soundscape cache: {len(sc_cache_meta)} windows")

# --- Build soundscape label matrix from ground truth ---
sc_labels_raw = pd.read_csv(LABELS_PATH).drop_duplicates()
sc_labels_raw["start_sec"] = pd.to_timedelta(sc_labels_raw["start"]).dt.total_seconds().astype(int)

Y_SC = np.zeros((len(sc_cache_meta), NUM_CLASSES), dtype=np.float32)
for i, row in sc_cache_meta.iterrows():
    matches = sc_labels_raw[
        (sc_labels_raw["filename"] == row["filename"]) &
        (sc_labels_raw["start_sec"] == row["start_sec"])
    ]
    for _, m in matches.iterrows():
        for lbl in str(m["primary_label"]).split(";"):
            lbl = lbl.strip()
            if lbl in LABEL2IDX:
                Y_SC[i, LABEL2IDX[lbl]] = 1.0

labeled_sc_mask = Y_SC.sum(axis=1) > 0
print(f"Soundscape labels: {labeled_sc_mask.sum()}/{len(Y_SC)} windows labeled, "
      f"{int(Y_SC.sum())} positives, {int((Y_SC.sum(axis=0) > 0).sum())} species")

# =================================================================
# FOLD ASSIGNMENT
# =================================================================

# --- Focal: StratifiedKFold by species ---
audio_for_split = audio_cache_meta.drop_duplicates("original_idx").reset_index(drop=True)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
audio_for_split["fold"] = -1
for fold, (_, val_idx) in enumerate(skf.split(audio_for_split, audio_for_split["primary_label"])):
    audio_for_split.loc[val_idx, "fold"] = fold
audio_cache_meta = audio_cache_meta.merge(
    audio_for_split[["original_idx", "fold"]], on="original_idx", how="left"
)
print(f"\nFocal fold distribution:\n{audio_cache_meta['fold'].value_counts().sort_index()}")

# --- Soundscape: file-level folds, all 66 files distributed ---
# All files including S22 participate in CV for maximum species coverage
# (46 multi-fold species vs 32 with S22 holdout). The non_s22_mask_sc
# filter in evaluation still excludes S22 windows from the primary metric.
from sklearn.model_selection import GroupKFold

sc_files = sc_cache_meta[["filename", "site"]].drop_duplicates().reset_index(drop=True)
gkf = GroupKFold(n_splits=N_FOLDS)
sc_files["fold"] = -1
for fold, (_, val_idx) in enumerate(gkf.split(sc_files, groups=sc_files["filename"])):
    sc_files.loc[sc_files.index[val_idx], "fold"] = fold

file_to_fold = dict(zip(sc_files["filename"], sc_files["fold"]))
sc_cache_meta["fold"] = sc_cache_meta["filename"].map(file_to_fold).fillna(-1).astype(int)
print(f"\nSoundscape fold distribution:")
print(sc_cache_meta["fold"].value_counts().sort_index())
# =================================================================
# UPSAMPLE RARE SPECIES
# =================================================================
counts = audio_cache_meta["primary_label"].value_counts()
rare_species = counts[counts < MIN_SAMPLE].index
extra_rows = []
for sp in rare_species:
    sp_rows = audio_cache_meta[audio_cache_meta["primary_label"] == sp]
    n_copies = int(np.ceil(MIN_SAMPLE / len(sp_rows))) - 1
    for _ in range(n_copies):
        extra_rows.append(sp_rows)

n_before = len(audio_cache_meta)
if extra_rows:
    audio_cache_meta = pd.concat([audio_cache_meta] + extra_rows, ignore_index=True)
print(f"\nUpsampled {len(rare_species)} rare species (min={MIN_SAMPLE}): "
      f"{n_before} -> {len(audio_cache_meta)} samples")

# Non-S22 mask for evaluation (S22 is a site with known label noise)
sc_sites = sc_cache_meta["site"].values
non_s22_mask_sc = sc_sites != "S22"
print(f"S22: {(~non_s22_mask_sc).sum()}, non-S22: {non_s22_mask_sc.sum()}")
print("OK Data loaded")


# =================================================================
# v2 ADDITION — Load pseudo CSV (soft labels on train_soundscapes)
# =================================================================
print(f"\nLoading pseudo CSV: {PSEUDO_CSV_PATH}")
t0 = time.time()
pseudo_df_full = pd.read_csv(PSEUDO_CSV_PATH)
print(f"  loaded {len(pseudo_df_full)} rows in {time.time()-t0:.1f}s")
print(f"  columns: {list(pseudo_df_full.columns[:5])} ... ({pseudo_df_full.shape[1]} total)")

# Identify label columns. Expected schema: filename, start_sec, end_sec, <234 species cols in PRIMARY_LABELS order>
_meta_cols = [c for c in pseudo_df_full.columns if c not in PRIMARY_LABELS]
_label_cols_in_csv = [c for c in pseudo_df_full.columns if c in PRIMARY_LABELS]
print(f"  meta cols: {_meta_cols}")
print(f"  matching label cols: {len(_label_cols_in_csv)} / {NUM_CLASSES}")
assert len(_label_cols_in_csv) == NUM_CLASSES, \
    f"pseudo CSV must contain all {NUM_CLASSES} species cols; got {len(_label_cols_in_csv)}"

# Reorder/select label columns to match PRIMARY_LABELS order strictly
Y_pseudo = pseudo_df_full[PRIMARY_LABELS].values.astype(np.float32)
print(f"  Y_pseudo: {Y_pseudo.shape}, range=[{Y_pseudo.min():.4f}, {Y_pseudo.max():.4f}], mean={Y_pseudo.mean():.4f}")

# Keep meta-only DataFrame for indexing
pseudo_meta = pseudo_df_full[_meta_cols].copy().reset_index(drop=True)
# Ensure required columns exist
for col in ["filename", "start_sec"]:
    assert col in pseudo_meta.columns, f"pseudo CSV missing required col '{col}'"
# Normalize filename column (ensure .ogg suffix and the file exists check at __getitem__ time)
pseudo_meta["filename"] = pseudo_meta["filename"].astype(str)
pseudo_meta["start_sec"] = pseudo_meta["start_sec"].astype(float)

# Quick sanity: first 3 rows
print(f"  pseudo_meta head:\n{pseudo_meta.head(3)}")
print(f"\nPseudo SC: {len(pseudo_meta)} rows on train_soundscapes")


In [ ]:
# =================================================================
# S3 -- EVAL UTILITIES + MEL TRANSFORM + SED MODELS
# =================================================================

def compute_macro_auc(y_true, y_pred, mask=None, class_mask=None):
    """Macro-averaged AUC across evaluable species."""
    if mask is not None:
        y_true, y_pred = y_true[mask], y_pred[mask]
    if class_mask is not None:
        y_true, y_pred = y_true[:, class_mask], y_pred[:, class_mask]
    aucs = []
    for c in range(y_true.shape[1]):
        col = y_true[:, c]
        if col.sum() == 0 or col.sum() == len(col):
            continue
        try:
            aucs.append(roc_auc_score(col, y_pred[:, c]))
        except ValueError:
            continue
    return (np.mean(aucs) if aucs else float("nan")), len(aucs)

def full_eval(y_true, y_pred, ns22, tm):
    r = {}
    a, n = compute_macro_auc(y_true, y_pred)
    r["macro_auc_all"], r["n_all"] = round(a, 4), n
    a, n = compute_macro_auc(y_true, y_pred, mask=ns22)
    r["non_s22_macro"], r["n_ns22"] = round(a, 4), n
    for t, cm in tm.items():
        a, n = compute_macro_auc(y_true, y_pred, mask=ns22, class_mask=cm)
        r[f"non_s22_{t}"] = round(a, 4)
    return r

# ------------------------------------------------------------------
# GPU Mel Spectrogram
# ------------------------------------------------------------------
class MelSpecTransform(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel_spec = torchaudio.transforms.MelSpectrogram(
            sample_rate=SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
            n_mels=N_MELS, f_min=FMIN, f_max=FMAX, power=2.0,
        )
        self.db_transform = torchaudio.transforms.AmplitudeToDB(top_db=80)

    def forward(self, waveform):
        return self.db_transform(self.mel_spec(waveform))

# ------------------------------------------------------------------
# GPU SpecAugment
# ------------------------------------------------------------------
class SpecAugment(nn.Module):
    def __init__(self):
        super().__init__()
        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=FREQ_MASK_PARAM)
        self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=TIME_MASK_PARAM)

    def forward(self, mel):
        for _ in range(NUM_FREQ_MASKS):
            mel = self.freq_mask(mel)
        for _ in range(NUM_TIME_MASKS):
            mel = self.time_mask(mel)
        return mel

# ------------------------------------------------------------------
# Frozen Perch teacher -- ONNX inference, no gradients
# ------------------------------------------------------------------
import onnxruntime as ort

class PerchTeacher:
    """Frozen Perch v2 via ONNX. Takes 5s waveforms, returns 1536-d embeddings.
    The teacher is never updated -- it provides a stable distillation target."""

    def __init__(self, onnx_path, device_str="cuda"):
        providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] \
            if device_str == "cuda" else ["CPUExecutionProvider"]
        self.session = ort.InferenceSession(str(onnx_path), providers=providers)
        self.input_name = self.session.get_inputs()[0].name
        self._out_names = [o.name for o in self.session.get_outputs()]
        self._embed_idx = None
        for i, o in enumerate(self.session.get_outputs()):
            if o.shape and o.shape[-1] == PERCH_EMBED_DIM:
                self._embed_idx = i
                break
        if self._embed_idx is None:
            self._embed_idx = 1
        print(f"Perch ONNX loaded: embed_idx={self._embed_idx}")

    @torch.no_grad()
    def embed(self, waveforms_5s):
        """waveforms_5s: (B, 160000) float32, returns (B, 1536) embeddings."""
        wav_np = waveforms_5s.cpu().numpy()
        results = self.session.run(None, {self.input_name: wav_np})
        return torch.from_numpy(results[self._embed_idx]).float()

# ------------------------------------------------------------------
# Distillation head: GAP + Linear to Perch embedding space
# ------------------------------------------------------------------
class DistillHead(nn.Module):
    """Projects backbone features to Perch's 1536-d space via GAP + Linear."""
    def __init__(self, backbone_dim, embed_dim=1536):
        super().__init__()
        self.proj = nn.Linear(backbone_dim, embed_dim)

    def forward(self, feature_map):
        gap = feature_map.mean(dim=[2, 3])   # (B, C, F, T) -> (B, C)
        return self.proj(gap)                 # (B, embed_dim)

# ------------------------------------------------------------------
# SED Model V2: GeMFreq + bottleneck + AttBlock (recommended)
# ------------------------------------------------------------------
class GeMFreqPool(nn.Module):
    """Generalized Mean pooling over frequency. Learnable p starts at 3.0
    (sharper than mean, softer than max). Lets the model emphasize
    frequency bands where species vocalize."""
    def __init__(self, p_init=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(float(p_init)))
        self.eps = eps

    def forward(self, x):
        p = self.p.clamp(min=1.0)
        x = x.clamp(min=self.eps).pow(p)
        x = x.mean(dim=2)
        return x.pow(1.0 / p)

class BirdSEDModel(nn.Module):
    """SED model with 1st-place-inspired design: https://www.kaggle.com/code/nikitababich/birdclef2025-1st-place-inference
    - GeMFreq pooling (learnable, sharper than mean)
    - 512-dim bottleneck with dropout
    - Attention-weighted clip logits from frame logits
    - Distillation: GAP+Linear branch for MSE to Perch
    - Stop gradient: backbone trains from distillation only
    """
    def __init__(self, backbone_name=BACKBONE_NAME, num_classes=NUM_CLASSES,
                 drop_path_rate=0.1, hidden_dim=512):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name, pretrained=True, in_chans=1,
            num_classes=0, global_pool="", drop_path_rate=drop_path_rate,
        )
        with torch.no_grad():
            n_tf = TRAIN_SAMPLES // HOP_LENGTH + 1
            dummy = torch.randn(1, 1, N_MELS, n_tf)
            feat = self.backbone(dummy)
            self.backbone_dim = feat.shape[1]
            print(f"V2 backbone: {tuple(feat.shape)}  (C={self.backbone_dim})")

        self.gem_freq = GeMFreqPool(p_init=3.0)
        self.dense = nn.Sequential(
            nn.Dropout(0.25),
            nn.Linear(self.backbone_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
        )
        self.att = nn.Conv1d(hidden_dim, num_classes, kernel_size=1, bias=True)
        self.cla = nn.Conv1d(hidden_dim, num_classes, kernel_size=1, bias=True)
        nn.init.xavier_uniform_(self.att.weight)
        nn.init.xavier_uniform_(self.cla.weight)
        self.att.bias.data.fill_(0.)
        self.cla.bias.data.fill_(0.)
        if USE_PERCH_DISTILL:
            self.distill_head = DistillHead(self.backbone_dim, PERCH_EMBED_DIM)

    def forward(self, x, return_framewise=False, return_distill=False):
        h = self.backbone(x)
        distill_emb = None
        if return_distill and hasattr(self, 'distill_head'):
            distill_emb = self.distill_head(h)

        # Stop gradient: SED head doesn't update the backbone
        h_cls = h.detach() if USE_PERCH_DISTILL else h

        h_cls = self.gem_freq(h_cls)            # (B, C, T)
        h_cls = h_cls.permute(0, 2, 1)          # (B, T, C)
        h_cls = self.dense(h_cls)               # (B, T, 512)
        h_cls = h_cls.permute(0, 2, 1)          # (B, 512, T)

        norm_att = torch.softmax(torch.tanh(self.att(h_cls)), dim=-1)
        framewise_logits = self.cla(h_cls)
        clip_logits = torch.sum(norm_att * framewise_logits, dim=2)

        fw = framewise_logits.permute(0, 2, 1) if return_framewise else None
        if return_framewise and return_distill: return clip_logits, fw, distill_emb
        elif return_framewise: return clip_logits, fw
        elif return_distill: return clip_logits, distill_emb
        return clip_logits

def make_model():
        return BirdSEDModel(BACKBONE_NAME).to(device)

print("OK Model definitions ready")


In [ ]:
# =================================================================
# S4 -- DATA PIPELINE
# =================================================================

def load_int16(path):
    """Load int16 waveform tensor to float32 in [-1, 1]."""
    waveform_int16 = torch.load(path, map_location="cpu")
    return waveform_int16.float() / 32767.0

_FC = {}
def load_focal(p):
    """Load focal waveform with simple LRU cache."""
    if p in _FC: return _FC[p]
    pp = WAVEFORM_CACHE_DIR / p
    if not pp.exists(): return None
    a = load_int16(pp).numpy()
    if len(_FC) >= 2000:
        _FC.pop(next(iter(_FC)))
    _FC[p] = a
    return a

_SC_CACHE = {}
def load_sc_waveform_from(cache_dir, cache_file):
    """Load a soundscape waveform with LRU cache."""
    key = str(cache_dir / cache_file)
    if key in _SC_CACHE: return _SC_CACHE[key]
    pp = cache_dir / cache_file
    if not pp.exists(): return None
    a = load_int16(pp).numpy()
    if len(_SC_CACHE) >= 200:
        _SC_CACHE.pop(next(iter(_SC_CACHE)))
    _SC_CACHE[key] = a
    return a

def extract_chunk_np(waveform, start_sample, n_samples):
    """Extract a chunk, left-padding if the recording is too short."""
    total = len(waveform)
    if total <= n_samples:
        return np.pad(waveform, (n_samples - total, 0))
    end = start_sample + n_samples
    if end > total:
        start_sample = max(0, total - n_samples)
    return waveform[start_sample:start_sample + n_samples]

def apply_aug(w):
    """Simple waveform augmentation: gain jitter + noise + shift."""
    if np.random.random() < AUG_PROB:
        w = w * (10 ** (np.random.uniform(*AUG_GAIN_DB_RANGE) / 20))
    if np.random.random() < AUG_PROB:
        sp = (w ** 2).mean()
        if sp > 1e-10:
            w = w + np.random.randn(*w.shape).astype(w.dtype) * np.sqrt(
                sp / (10 ** (np.random.uniform(*AUG_NOISE_SNR_DB_RANGE) / 10)))
    return w

# ------------------------------------------------------------------
# Build soundscape MixUp pool (labeled windows only)
# ------------------------------------------------------------------
sc_mixup_sources = []
_sc_file_meta = pd.read_csv(WAVEFORM_CACHE_DIR / "soundscape_file_meta.csv")
_sc_file_dict = dict(zip(_sc_file_meta["filename"], _sc_file_meta["cache_file"]))
_labeled_rows = []
for i in range(len(sc_cache_meta)):
    row = sc_cache_meta.iloc[i]
    if Y_SC[i].sum() > 0:
        cf = _sc_file_dict.get(row["filename"])
        if cf is not None:
            _labeled_rows.append({
                "filename": row["filename"], "start_sec": int(row["start_sec"]),
                "cache_file": cf, "label_idx": i, "fold": int(row.get("fold", -1)),
            })
if _labeled_rows:
    _labeled_meta = pd.DataFrame(_labeled_rows)
    sc_mixup_sources.append((WAVEFORM_CACHE_DIR, _labeled_meta, Y_SC))
    print(f"SC MixUp pool: {len(_labeled_meta)} labeled windows")

# ------------------------------------------------------------------
# FocalDS -- with Focal-Focal AND Focal-Soundscape MixUp
# ------------------------------------------------------------------
class FocalDS(Dataset):
    """Focal recording dataset. Returns (waveform, label, weight, mask, source_tag)."""
    def __init__(self, df, l2i, secondary_lookup=None,
                 sc_mixup_sources=None, fold_k=None, aug=False):
        self.df, self.l2i, self.aug = df.reset_index(drop=True), l2i, aug
        self.secondary_lookup = secondary_lookup
        self.sc_mixup_sources = sc_mixup_sources
        self.fold_k = fold_k

    def __len__(self): return len(self.df)

    def _load_chunk(self, r):
        w = load_focal(r["cache_file"])
        if w is None: return None, None
        if self.aug:
            start = np.random.randint(0, max(1, len(w) - TRAIN_SAMPLES + 1)) if len(w) > TRAIN_SAMPLES else 0
        else:
            start = int(r.get("start_sec", 0)) * SR
        ch = extract_chunk_np(w, start, TRAIN_SAMPLES)
        lb = np.zeros(NUM_CLASSES, dtype=np.float32)
        if str(r["primary_label"]) in self.l2i:
            lb[self.l2i[str(r["primary_label"])]] = 1.0
        if self.secondary_lookup is not None and "original_idx" in self.df.columns:
            for s in self.secondary_lookup.get(int(r["original_idx"]), []):
                if s in self.l2i: lb[self.l2i[s]] = 1.0
        return ch, lb

    def __getitem__(self, i):
        r1 = self.df.iloc[i]
        ch1, lb1 = self._load_chunk(r1)
        if ch1 is None:
            return (torch.zeros(1, TRAIN_SAMPLES), torch.zeros(NUM_CLASSES),
                    torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "focal_missing")

        # Focal-Focal MixUp
        if USE_FOCAL_MIXUP and self.aug and np.random.random() < MIXUP_PROB:
            ch2 = None
            for _ in range(3):
                j = np.random.randint(len(self.df))
                ch2, lb2 = self._load_chunk(self.df.iloc[j])
                if ch2 is not None: break
            if ch2 is not None:
                lam = np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA)
                ch_mix = (lam * ch1 + (1 - lam) * ch2).astype(np.float32)
                if self.aug: ch_mix = apply_aug(ch_mix)
                lb = np.maximum(lb1, lb2) if MIXUP_HARD else (lam * lb1 + (1 - lam) * lb2)
                return (torch.from_numpy(ch_mix).unsqueeze(0), torch.from_numpy(lb),
                        torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "focal")

        # Focal-Soundscape MixUp
        if (USE_FOCAL_SC_MIXUP and self.aug and self.sc_mixup_sources
                and np.random.random() < FOCAL_SC_MIXUP_PROB):
            src_idx = np.random.randint(len(self.sc_mixup_sources))
            cache_dir, meta_df_sc, labels = self.sc_mixup_sources[src_idx]
            eligible = meta_df_sc[meta_df_sc["fold"] != self.fold_k] if self.fold_k is not None else meta_df_sc
            if len(eligible) > 0:
                sc_row = eligible.iloc[np.random.randint(len(eligible))]
                sc_wav = load_sc_waveform_from(cache_dir, sc_row["cache_file"])
                if sc_wav is not None and len(sc_wav) >= TRAIN_SAMPLES:
                    sc_chunk = extract_chunk_np(sc_wav, int(sc_row["start_sec"]) * SR, TRAIN_SAMPLES)
                    lam = np.random.beta(FOCAL_SC_MIXUP_ALPHA, FOCAL_SC_MIXUP_ALPHA)
                    ch_mix = (lam * ch1 + (1 - lam) * sc_chunk).astype(np.float32)
                    if self.aug: ch_mix = apply_aug(ch_mix)
                    lb_sc = labels[int(sc_row["label_idx"])].astype(np.float32)
                    lb = np.maximum(lb1, lb_sc) if MIXUP_HARD else lam * lb1 + (1 - lam) * lb_sc
                    return (torch.from_numpy(ch_mix).unsqueeze(0), torch.from_numpy(lb),
                            torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "focal")

        # No MixUp
        if self.aug: ch1 = apply_aug(ch1)
        return (torch.from_numpy(ch1.astype(np.float32)).unsqueeze(0),
                torch.from_numpy(lb1),
                torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "focal")

# ------------------------------------------------------------------
# ScDS -- Labeled soundscape windows
# ------------------------------------------------------------------
class ScDS(Dataset):
    def __init__(self, Y, sc_df, aug=False):
        self.Y, self.df, self.aug = Y, sc_df.reset_index(drop=True), aug
    def __len__(self): return len(self.Y)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        wav_full = load_sc_waveform_from(WAVEFORM_CACHE_DIR, row.get("cache_file")) \
                   if row.get("cache_file") else None
        if wav_full is None:
            wav_t = torch.zeros(1, TRAIN_SAMPLES)
        else:
            chunk = extract_chunk_np(wav_full, int(row["start_sec"]) * SR, TRAIN_SAMPLES)
            if self.aug: chunk = apply_aug(chunk)
            wav_t = torch.from_numpy(chunk.astype(np.float32)).unsqueeze(0)
        return (wav_t, torch.from_numpy(self.Y[i].astype(np.float32)),
                torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "sc")

# ------------------------------------------------------------------
# Load focal secondary labels
# ------------------------------------------------------------------
focal_secondary_labels = None
if USE_FOCAL_SECONDARY:
    focal_secondary_labels = {}
    for idx, row in train_df.iterrows():
        sec = row.get("secondary_labels", "")
        if pd.isna(sec) or sec in ("", "[]"): continue
        try:
            sec_list = eval(sec) if isinstance(sec, str) else []
        except: continue
        valid = [s for s in sec_list if s in LABEL2IDX]
        if valid: focal_secondary_labels[idx] = valid
    print(f"Focal secondary labels: {len(focal_secondary_labels)} files")

# ------------------------------------------------------------------
# Multi-source batch sampler
# ------------------------------------------------------------------
class MixSamp(torch.utils.data.Sampler):
    """Controls batch composition via per-source shares."""
    def __init__(self, sizes, names, shares, bs, nst, seed=0):
        self.sizes, self.names, self.bs, self.nst = sizes, names, bs, nst
        self.rng = np.random.default_rng(seed)
        per_src = [max(1, int(round(bs * shares.get(n, 0.0)))) for n in names]
        total = sum(per_src)
        if total != bs:
            per_src[int(np.argmax(per_src))] += (bs - total)
        self.per_src = per_src
        self.offsets = [0]
        for s in sizes[:-1]:
            self.offsets.append(self.offsets[-1] + s)
    def __len__(self): return self.nst
    def __iter__(self):
        for _ in range(self.nst):
            batch = []
            for off, size, n in zip(self.offsets, self.sizes, self.per_src):
                if n <= 0 or size <= 0: continue
                idxs = self.rng.integers(0, size, size=n)
                batch.extend([off + int(i) for i in idxs])
            self.rng.shuffle(batch)
            yield batch

def collate_m(batch):
    return (torch.stack([b[0] for b in batch]),
            torch.stack([b[1] for b in batch]),
            torch.stack([b[2] for b in batch]),
            torch.stack([b[3] for b in batch]),
            [b[4] for b in batch])

def mk_sw(sr):
    """Per-sample source weight tensor."""
    return torch.tensor([SOURCE_WEIGHTS.get(s, 0.0) for s in sr], dtype=torch.float32)

print("OK Data pipeline ready")


# =================================================================
# v2 ADDITION -- PseudoScDS: read .ogg directly with soundfile
# =================================================================
import soundfile as sf

class PseudoScDS(Dataset):
    """Pseudo-labeled soundscape windows. Reads .ogg directly with soundfile,
    seeking to start_sec for efficiency.

    Returns (waveform, soft_label, weight, mask, source_tag).
    soft_label is float32 in [0, 1] (post NB1 gamma correction).
    """
    def __init__(self, meta_df, Y_soft, audio_dir, aug=False):
        self.meta = meta_df.reset_index(drop=True)
        self.Y = Y_soft.astype(np.float32)
        self.audio_dir = Path(audio_dir)
        self.aug = aug
        self.filenames = self.meta["filename"].values
        self.start_secs = self.meta["start_sec"].values

    def __len__(self):
        return len(self.meta)

    def __getitem__(self, i):
        fn = self.filenames[i]
        if not fn.endswith(".ogg"):
            fn = fn + ".ogg"
        path = self.audio_dir / fn
        start_sec = float(self.start_secs[i])

        try:
            wav, sr = sf.read(
                str(path),
                start=int(start_sec * SR),
                frames=TRAIN_SAMPLES,
                dtype="float32",
                always_2d=False,
            )
            if wav.ndim > 1:
                wav = wav.mean(axis=1)
            if sr != SR:
                import librosa
                wav = librosa.resample(wav, orig_sr=sr, target_sr=SR)
            if len(wav) < TRAIN_SAMPLES:
                wav = np.pad(wav, (0, TRAIN_SAMPLES - len(wav)))
            else:
                wav = wav[:TRAIN_SAMPLES]
        except Exception:
            wav = np.zeros(TRAIN_SAMPLES, dtype=np.float32)

        if self.aug:
            wav = apply_aug(wav)

        wav_t = torch.from_numpy(wav.astype(np.float32)).unsqueeze(0)  # (1, T)
        soft_lb = torch.from_numpy(self.Y[i])
        return (wav_t, soft_lb,
                torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "pseudo_sc")


print("OK PseudoScDS defined (reads .ogg via soundfile.read with seek)")


In [ ]:
# ============================================================
# Cell 8: build_active_datasets v2 — adds pseudo_sc as 3rd source
# ============================================================

def build_active_datasets(fold_k):
    items = []
    if USE_FOCAL:
        fds = FocalDS(audio_cache_meta[audio_cache_meta["fold"] != fold_k],
                      LABEL2IDX, secondary_lookup=focal_secondary_labels,
                      sc_mixup_sources=sc_mixup_sources if USE_FOCAL_SC_MIXUP else None,
                      fold_k=fold_k, aug=True)
        items.append(("focal", fds, len(fds)))
    if USE_LABELED_SC:
        vm = sc_cache_meta["fold"].values == fold_k
        sc_train_df = sc_cache_meta[~vm].reset_index(drop=True)
        Y_tr = Y_SC[~vm]
        sds = ScDS(Y_tr, sc_train_df, aug=True)
        items.append(("sc", sds, len(sds)))
    if USE_PSEUDO_SC:
        # Pseudo SC: not fold-filtered (unlabeled, no leakage concern)
        pds = PseudoScDS(pseudo_meta, Y_pseudo, TRAIN_SC_DIR, aug=True)
        items.append(("pseudo_sc", pds, len(pds)))
    return items

print("OK build_active_datasets v2 ready (focal + sc + pseudo_sc)")


In [ ]:
# ============================================================
# Cell 9: .ogg patch — replace load_focal/load_sc to read .ogg directly from /content/data
# ============================================================
# v2 (Drive-based): Tucker .pt cache 廃止、cache_file → 元 .ogg をマッピング
import soundfile as sf
import numpy as np
import pandas as pd
import torch
from pathlib import Path

TRAIN_AUDIO_DIR = Path("/content/data/train_audio")

# Build lookup: cache_file (Tucker .pt name) → original .ogg filename
print("Building cache_file → .ogg lookup ...")
_CACHE_FILE_TO_OGG_FOCAL = {row["cache_file"]: row["filename"] for _, row in audio_cache_meta.iterrows()}
_CACHE_FILE_TO_OGG_SC = {row["cache_file"]: row["filename"] for _, row in sc_cache_meta.iterrows()}
print(f"  focal lookup: {len(_CACHE_FILE_TO_OGG_FOCAL)} entries")
print(f"  sc lookup:    {len(_CACHE_FILE_TO_OGG_SC)} entries")

# Patch load_focal: read .ogg from train_audio
def load_focal(p):
    # Read .ogg from /content/data/train_audio. p is cache_file (audio/audio_xxx.pt)
    if p in _FC: return _FC[p]
    ogg_rel = _CACHE_FILE_TO_OGG_FOCAL.get(p)
    if ogg_rel is None:
        return None
    ogg_path = TRAIN_AUDIO_DIR / ogg_rel
    if not ogg_path.exists():
        return None
    try:
        wav, sr = sf.read(str(ogg_path), dtype="int16")
        if wav.ndim > 1:
            wav = wav.mean(axis=1).astype("int16")
        if sr != SR:
            import librosa
            wav = librosa.resample(wav.astype("float32"), orig_sr=sr, target_sr=SR)
            wav = wav.astype("float32") / 32767.0   # already float
            a = wav
        else:
            a = wav.astype("float32") / 32767.0   # int16 → [-1, 1]
    except Exception:
        return None
    if len(_FC) >= 2000:
        _FC.pop(next(iter(_FC)))
    _FC[p] = a
    return a

# Patch load_sc_waveform_from: read .ogg from train_soundscapes
TRAIN_SC_DIR_LOCAL = Path("/content/data/train_soundscapes")
def load_sc_waveform_from(cache_dir, cache_file):
    # Read .ogg from /content/data/train_soundscapes. cache_dir ignored (kept for API compat)
    key = str(cache_dir) + "::" + str(cache_file)
    if key in _SC_CACHE: return _SC_CACHE[key]
    ogg_rel = _CACHE_FILE_TO_OGG_SC.get(cache_file)
    if ogg_rel is None:
        return None
    ogg_path = TRAIN_SC_DIR_LOCAL / ogg_rel
    if not ogg_path.exists():
        return None
    try:
        wav, sr = sf.read(str(ogg_path), dtype="int16")
        if wav.ndim > 1:
            wav = wav.mean(axis=1).astype("int16")
        if sr != SR:
            import librosa
            wav = librosa.resample(wav.astype("float32"), orig_sr=sr, target_sr=SR)
            a = wav.astype("float32")
        else:
            a = wav.astype("float32") / 32767.0
    except Exception:
        return None
    if len(_SC_CACHE) >= 200:
        _SC_CACHE.pop(next(iter(_SC_CACHE)))
    _SC_CACHE[key] = a
    return a

# Sanity check
print("\nSanity check load_focal:")
cf0 = audio_cache_meta.iloc[0]["cache_file"]
sample = load_focal(cf0)
if sample is not None:
    print(f"  {cf0} -> .ogg={_CACHE_FILE_TO_OGG_FOCAL[cf0]}, "
          f"shape={sample.shape}, min={sample.min():.4f}, max={sample.max():.4f}")
else:
    print(f"  FAILED to load {cf0}")

print("\nSanity check load_sc_waveform_from:")
cf_sc = sc_cache_meta.iloc[0]["cache_file"]
sample_sc = load_sc_waveform_from(None, cf_sc)
if sample_sc is not None:
    print(f"  {cf_sc} -> .ogg={_CACHE_FILE_TO_OGG_SC[cf_sc]}, "
          f"shape={sample_sc.shape}, min={sample_sc.min():.4f}, max={sample_sc.max():.4f}")
else:
    print(f"  FAILED to load {cf_sc}")

# Clear caches (in case prior loader populated)
_FC.clear()
_SC_CACHE.clear()
print("\nload_focal / load_sc_waveform_from patched -> reads .ogg directly")


In [ ]:
# =================================================================
# S5 -- TRAINING
# =================================================================

def _load_val_waveforms(val_sc_df):
    """Load validation waveforms (always 5s)."""
    sc_file_meta = pd.read_csv(WAVEFORM_CACHE_DIR / "soundscape_file_meta.csv")
    sc_file_dict = dict(zip(sc_file_meta["filename"], sc_file_meta["cache_file"]))
    wavs = []
    for _, row in val_sc_df.iterrows():
        cf = sc_file_dict.get(row["filename"])
        if cf is not None:
            w = load_sc_waveform_from(WAVEFORM_CACHE_DIR, cf)
            if w is not None:
                chunk = extract_chunk_np(w, int(row["start_sec"]) * SR, VAL_SAMPLES)
                wavs.append(torch.from_numpy(chunk.astype(np.float32)).unsqueeze(0))
            else: wavs.append(torch.zeros(1, VAL_SAMPLES))
        else: wavs.append(torch.zeros(1, VAL_SAMPLES))
    return wavs

def _predict_from_waveforms(model, mel_transform, wav_list, batch_size=64):
    """Inference: mel -> model -> sigmoid. Distillation head is NOT used."""
    model.eval()
    preds_clip, preds_fmax, preds_blend = [], [], []
    with torch.no_grad():
        for s in range(0, len(wav_list), batch_size):
            batch = torch.stack(wav_list[s:s+batch_size]).to(device)
            mel = mel_transform(batch)
            B = mel.size(0)
            for i in range(B):
                mel[i] = (mel[i] - mel[i].mean()) / (mel[i].std() + 1e-6)
            with autocast():
                clip_logits, framewise = model(mel, return_framewise=True)
                frame_max = framewise.max(dim=1).values
                p_clip = torch.sigmoid(clip_logits).cpu().numpy()
                p_fmax = torch.sigmoid(frame_max).cpu().numpy()
                p_blend = 0.5 * p_clip + 0.5 * p_fmax
            preds_clip.append(p_clip); preds_fmax.append(p_fmax); preds_blend.append(p_blend)
    return {"clip": np.concatenate(preds_clip), "fmax": np.concatenate(preds_fmax),
            "blend": np.concatenate(preds_blend)}

def build_active_datasets(fold_k):
    items = []
    if USE_FOCAL:
        fds = FocalDS(audio_cache_meta[audio_cache_meta["fold"] != fold_k],
                      LABEL2IDX, secondary_lookup=focal_secondary_labels,
                      sc_mixup_sources=sc_mixup_sources if USE_FOCAL_SC_MIXUP else None,
                      fold_k=fold_k, aug=True)
        items.append(("focal", fds, len(fds)))
    if USE_LABELED_SC:
        vm = sc_cache_meta["fold"].values == fold_k
        sc_train_df = sc_cache_meta[~vm].reset_index(drop=True)
        Y_tr = Y_SC[~vm]
        sds = ScDS(Y_tr, sc_train_df, aug=True)
        items.append(("sc", sds, len(sds)))
    return items

def train_fold(fold_k):
    vm = sc_cache_meta["fold"].values == fold_k
    Y_val = Y_SC[vm]
    ns22_val = non_s22_mask_sc[vm]
    val_sc_df = sc_cache_meta[vm].reset_index(drop=True)

    active = build_active_datasets(fold_k)
    names, datasets, sizes = zip(*active)
    mds = ConcatDataset(list(datasets))
    nst = max(100, int(sum(sizes) / BATCH))

    print(f"  Streams: {dict(zip(names, sizes))}  steps/ep: {nst}")

    m = make_model()
    mel_transform = MelSpecTransform().to(device)
    spec_augment = SpecAugment().to(device)
    perch_teacher = PerchTeacher(PERCH_ONNX_PATH,
                                  "cuda" if torch.cuda.is_available() else "cpu") \
                    if USE_PERCH_DISTILL else None

    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=WD)
    scaler = GradScaler()
    warmup_steps = nst * WARMUP_EPOCHS
    total_steps  = nst * EPOCHS
    warmup_sched = torch.optim.lr_scheduler.LinearLR(opt, start_factor=1/25, end_factor=1.0,
                                                      total_iters=warmup_steps)
    cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total_steps - warmup_steps,
                                                               eta_min=1e-6)
    sch = torch.optim.lr_scheduler.SequentialLR(opt, schedulers=[warmup_sched, cosine_sched],
                                                 milestones=[warmup_steps])

    history = {"ep": [], "train_loss": [], "cls_loss": [], "dist_loss": [],
               "macro": [], "ns22_macro": [],
               "ns22_Aves": [], "ns22_Amphibia": [], "ns22_Insecta": [], "ns22_Mammalia": [],
               "val_preds": []}
    best_ns22, best_state_ns22 = -1.0, None
    best_macro, best_state_macro = -1.0, None
    val_wavs = _load_val_waveforms(val_sc_df)

    for ep in range(EPOCHS):
        m.train()
        smp = MixSamp(list(sizes), list(names), SHARES, BATCH, nst, seed=42 + ep)
        tl = DataLoader(mds, batch_sampler=smp, collate_fn=collate_m,
                        num_workers=0, pin_memory=True)
        el, el_cls, el_dist, nb_count = 0.0, 0.0, 0.0, 0
        t0 = time.time()

        for wav, lb, wt, mk, sr in tl:
            wav, lb, wt, mk = wav.to(device), lb.to(device), wt.to(device), mk.to(device)
            sw = mk_sw(sr).to(device)

            with torch.no_grad():
                mel = mel_transform(wav)
                B = mel.size(0)
                for i in range(B):
                    mel[i] = (mel[i] - mel[i].mean()) / (mel[i].std() + 1e-6)
                mel = spec_augment(mel)

            with autocast():
                if USE_PERCH_DISTILL:
                    clip_logits, framewise, distill_emb = m(mel, return_framewise=True,
                                                            return_distill=True)
                else:
                    clip_logits, framewise = m(mel, return_framewise=True)

                frame_max_logits = framewise.max(dim=1).values

                # Classification loss
                bce_clip = F.binary_cross_entropy_with_logits(clip_logits, lb, reduction="none")
                bce_frame = F.binary_cross_entropy_with_logits(frame_max_logits, lb, reduction="none")
                bce = 0.5 * bce_clip + 0.5 * bce_frame
                ps = (bce * wt * mk).sum(1) / (mk.sum(1) + 1e-8)
                cls_loss = (ps * sw).mean()

                # Distillation loss
                if USE_PERCH_DISTILL and perch_teacher is not None:
                    with torch.no_grad():
                        wav_5s = wav.squeeze(1)
                        N = wav_5s.shape[1]
                        if N > 160000:
                            start = (N - 160000) // 2
                            wav_5s = wav_5s[:, start:start + 160000]
                        elif N < 160000:
                            wav_5s = F.pad(wav_5s, (0, 160000 - N))
                        perch_emb = perch_teacher.embed(wav_5s).to(device)
                    distill_loss = F.mse_loss(distill_emb, perch_emb)
                    loss = cls_loss + ALPHA_DISTILL * distill_loss
                else:
                    distill_loss = torch.tensor(0.0)
                    loss = cls_loss

            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sch.step()
            el += loss.item(); el_cls += cls_loss.item()
            el_dist += distill_loss.item(); nb_count += 1

        # Validation
        val_preds_dict = _predict_from_waveforms(m, mel_transform, val_wavs)
        val_preds = val_preds_dict["blend"]
        r = full_eval(Y_val, val_preds, ns22_val, TAXON_MASKS)
        for mode in ["clip", "fmax", "blend"]:
            r_mode = full_eval(Y_val, val_preds_dict[mode], ns22_val, TAXON_MASKS)
            r[f"ns22_{mode}"] = r_mode["non_s22_macro"]

        history["ep"].append(ep)
        history["train_loss"].append(round(el / nb_count, 5))
        history["cls_loss"].append(round(el_cls / nb_count, 5))
        history["dist_loss"].append(round(el_dist / nb_count, 5))
        history["macro"].append(r["macro_auc_all"])
        history["ns22_macro"].append(r["non_s22_macro"])
        for t in ["Aves", "Amphibia", "Insecta", "Mammalia"]:
            history[f"ns22_{t}"].append(r[f"non_s22_{t}"])
        history["val_preds"].append(val_preds.astype(np.float32))

        tag_ns22 = ""; tag_macro = ""
        if r["non_s22_macro"] > best_ns22:
            best_ns22 = r["non_s22_macro"]
            best_state_ns22 = {k: v.cpu().clone() for k, v in m.state_dict().items()}
            tag_ns22 = " *ns22"
        if r["macro_auc_all"] > best_macro:
            best_macro = r["macro_auc_all"]
            best_state_macro = {k: v.cpu().clone() for k, v in m.state_dict().items()}
            tag_macro = " *macro"

        dist_str = f" dist={el_dist/nb_count:.4f}" if USE_PERCH_DISTILL else ""
        print(f"    Ep{ep:02d}: loss={el/nb_count:.4f} cls={el_cls/nb_count:.4f}{dist_str} "
              f"lr={opt.param_groups[0]['lr']:.1e} | "
              f"ns22: {r['ns22_blend']:.4f} | "
              f"Av={r['non_s22_Aves']:.4f} Am={r['non_s22_Amphibia']:.4f} "
              f"In={r['non_s22_Insecta']:.4f} Ma={r['non_s22_Mammalia']:.4f} "
              f"[{time.time()-t0:.0f}s]{tag_ns22}{tag_macro}")

    del perch_teacher, m, mel_transform, spec_augment
    torch.cuda.empty_cache(); gc.collect()
    return best_state_ns22, best_state_macro, history

print("OK Training function ready")


In [ ]:
# ============================================================
# Cell 11: Drive mirror — auto-mirror checkpoints after each fold
# ============================================================
import shutil
from pathlib import Path

DRIVE_OUT_DIR = OUT_DIR_DRIVE
_orig_train_fold = train_fold

def train_fold(fold_k):
    print(f"=== train_fold({fold_k}) ===")
    # Pre-load any existing ckpts from Drive (resume support)
    for fname in [f"r1v2_fold{fold_k}_best_ns22.pth",
                  f"r1v2_fold{fold_k}_best_macro.pth",
                  f"r1v2_fold{fold_k}_history.json"]:
        d = DRIVE_OUT_DIR / fname
        l = OUT_DIR / fname
        if d.exists() and not l.exists():
            shutil.copy(str(d), str(l))
            print(f"  pre-loaded {fname} from Drive")

    result = _orig_train_fold(fold_k)

    # Mirror new outputs back to Drive
    for src in OUT_DIR.glob(f"r1v2_fold{fold_k}_*"):
        dst = DRIVE_OUT_DIR / src.name
        shutil.copy2(str(src), str(dst))
        print(f"  mirrored {src.name} -> Drive ({dst.stat().st_size/1e6:.1f}MB)")
    return result

print("OK Drive-mirroring train_fold ready")


In [ ]:
# ============================================================
# Cell 12: Fold loop — train each fold, save ckpt + history
# ============================================================
import json as _json

if MODE != "train":
    print("Skipping training (MODE='infer')")
    oof_ns22 = None
    all_hist = {}
else:
    oof_ns22 = np.full((len(sc_cache_meta), NUM_CLASSES), np.nan, dtype=np.float32)
    all_hist = {}
    for fold_k in FOLDS:
        print(f"\n{'='*60}")
        print(f"FOLD {fold_k}  (R1 v2)")
        print(f"{'='*60}")
        torch.manual_seed(SEED + fold_k)
        np.random.seed(SEED + fold_k)
        random.seed(SEED + fold_k)

        vm = sc_cache_meta["fold"].values == fold_k
        val_sc_df_k = sc_cache_meta[vm].reset_index(drop=True)

        best_ns22_state, best_macro_state, hist = train_fold(fold_k)
        all_hist[fold_k] = hist

        # Save PyTorch checkpoints (.pth)
        if best_ns22_state is not None:
            torch.save(best_ns22_state, OUT_DIR / f"r1v2_fold{fold_k}_best_ns22.pth")
            print(f"  saved best_ns22 (auc={max(hist['ns22_macro']):.4f})")
        if best_macro_state is not None:
            torch.save(best_macro_state, OUT_DIR / f"r1v2_fold{fold_k}_best_macro.pth")
            print(f"  saved best_macro (auc={max(hist['macro']):.4f})")

        # Save history (without val_preds to keep size small)
        hist_serializable = {k: (v if k != "val_preds" else f"<{len(v)} arrays omitted>")
                             for k, v in hist.items()}
        with open(OUT_DIR / f"r1v2_fold{fold_k}_history.json", "w") as f:
            _json.dump(hist_serializable, f, indent=2, default=str)

        # OOF blend prediction with best_macro
        if best_macro_state is not None:
            mel_tf = MelSpecTransform().to(device)
            val_wavs_k = _load_val_waveforms(val_sc_df_k)
            m_oof = make_model()
            m_oof.load_state_dict(best_macro_state, strict=False)
            oof_ns22[vm] = _predict_from_waveforms(m_oof, mel_tf, val_wavs_k)["blend"]
            del m_oof
            torch.cuda.empty_cache()

print("\nFold loop done")


In [ ]:
# ============================================================
# Cell 13: ONNX export — clip + framewise outputs (no distill head)
# ============================================================
import onnxruntime as ort

if MODE != "train":
    print("Skipping ONNX export (MODE='infer')")
else:
    INF_N_MELS = N_MELS  # full mel; Tucker used 128 in original; we keep N_MELS for accuracy
    INF_N_FRAMES = VAL_SAMPLES // HOP_LENGTH + 1

    class SEDExportWrapper(nn.Module):
        # Inference-only wrapper: takes mel, returns (clip_logits, framewise_logits)
        # with framewise transposed to (B, T, num_classes) for downstream blend NB.
        def __init__(self, backbone_name, num_classes, backbone_dim, hidden_dim=512):
            super().__init__()
            self.backbone = timm.create_model(
                backbone_name, pretrained=False, in_chans=1,
                num_classes=0, global_pool="", drop_path_rate=0.1,
            )
            self.gem_freq = GeMFreqPool(p_init=3.0)
            self.dense_drop1 = nn.Dropout(0.25)
            self.dense_conv = nn.Conv1d(backbone_dim, hidden_dim, 1)
            self.dense_relu = nn.ReLU(inplace=True)
            self.dense_drop2 = nn.Dropout(0.5)
            self.att = nn.Conv1d(hidden_dim, num_classes, 1)
            self.cla = nn.Conv1d(hidden_dim, num_classes, 1)

        def forward(self, mel):
            h = self.backbone(mel)
            h = self.gem_freq(h)
            h = self.dense_drop1(h)
            h = self.dense_conv(h)
            h = self.dense_relu(h)
            h = self.dense_drop2(h)
            norm_att = torch.softmax(torch.tanh(self.att(h)), dim=-1)
            framewise = self.cla(h)
            clip = torch.sum(norm_att * framewise, dim=2)
            return clip, framewise.permute(0, 2, 1)

    def load_and_remap_state(export_model, trained_state):
        remap = {}
        for k, v in trained_state.items():
            if k.startswith("distill_head."):
                continue
            if k == "dense.1.weight":
                remap["dense_conv.weight"] = v.unsqueeze(-1)
            elif k == "dense.1.bias":
                remap["dense_conv.bias"] = v
            else:
                remap[k] = v
        export_model.load_state_dict(remap, strict=False)

    for fold_k in FOLDS:
        ckpt_path = OUT_DIR / f"r1v2_fold{fold_k}_best_macro.pth"
        if not ckpt_path.exists():
            print(f"  skip fold {fold_k}: ckpt missing")
            continue
        print(f"\nExporting fold {fold_k}...")
        state = torch.load(str(ckpt_path), map_location="cpu")

        # Determine backbone_dim from a reference model
        m_ref = make_model()
        backbone_dim = m_ref.backbone_dim
        del m_ref
        torch.cuda.empty_cache()

        export_model = SEDExportWrapper(BACKBONE_NAME, NUM_CLASSES, backbone_dim).to(device)
        load_and_remap_state(export_model, state)
        export_model.eval()

        dummy_mel = torch.randn(1, 1, INF_N_MELS, INF_N_FRAMES).to(device)
        onnx_path = OUT_DIR / f"r1v2_fold{fold_k}.onnx"
        torch.onnx.export(
            export_model, dummy_mel, str(onnx_path),
            input_names=["mel"],
            output_names=["clip_logits", "framewise_logits"],
            dynamic_axes={"mel": {0: "batch"},
                          "clip_logits": {0: "batch"},
                          "framewise_logits": {0: "batch"}},
            opset_version=14,
        )

        # Verify
        sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
        onx_out = sess.run(None, {"mel": dummy_mel.cpu().numpy()})
        with torch.no_grad():
            ref_clip, _ = export_model(dummy_mel)
        diff = np.abs(ref_clip.cpu().numpy() - onx_out[0]).max()
        print(f"  ONNX verify: max|diff|={diff:.3e}")
        assert diff < 1e-3, f"ONNX export diverged: {diff}"
        del sess

        size_mb = onnx_path.stat().st_size / 1e6
        print(f"  exported {onnx_path.name} ({size_mb:.1f} MB)")
        del export_model
        torch.cuda.empty_cache()

    print("\nONNX export done")


In [ ]:
# ============================================================
# Cell 14: Mirror all final outputs to Drive
# ============================================================
import shutil

for src in OUT_DIR.glob("r1v2_*"):
    dst = DRIVE_OUT_DIR / src.name
    shutil.copy2(str(src), str(dst))
    print(f"  mirrored {src.name} -> Drive ({dst.stat().st_size/1e6:.1f}MB)")

print("\nFinal Drive contents:")
for p in sorted(DRIVE_OUT_DIR.iterdir()):
    print(f"  {p.name} ({p.stat().st_size/1e6:.1f}MB)")


In [ ]:
# ============================================================
# Cell 15 (optional): Upload ONNX bundle to Kaggle Dataset
# ============================================================
import json as _json, shutil, tempfile
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi(); api.authenticate()

DATASET_USER  = "maekeso"
DATASET_SLUG  = "birdclef2026-exp013-r1-v2-student-sed"
DATASET_TITLE = "BirdCLEF2026 exp013 R1 v2 Student SED"

bundle = Path("/content/exp013_r1v2_onnx_bundle")
if bundle.exists(): shutil.rmtree(bundle)
bundle.mkdir(parents=True)

n_copied = 0
for src in DRIVE_OUT_DIR.glob("r1v2_fold*.onnx"):
    shutil.copy2(str(src), str(bundle / src.name))
    n_copied += 1
    # External data file (.onnx.data) for large models
    extdata = src.with_suffix(src.suffix + ".data")
    if extdata.exists():
        shutil.copy2(str(extdata), str(bundle / extdata.name))

if n_copied == 0:
    print("No ONNX files to upload; skipping")
else:
    (bundle / "dataset-metadata.json").write_text(_json.dumps({
        "title": DATASET_TITLE,
        "id":    f"{DATASET_USER}/{DATASET_SLUG}",
        "licenses": [{"name": "CC0-1.0"}],
    }, indent=2))

    print(f"Uploading {n_copied} ONNX files...")
    try:
        api.dataset_create_new(folder=str(bundle), dir_mode="skip",
                               public=False, quiet=False)
        print(f"Created dataset: {DATASET_USER}/{DATASET_SLUG}")
    except Exception as e:
        print(f"create_new failed: {str(e)[:200]}")
        try:
            api.dataset_create_version(folder=str(bundle),
                                        version_notes="exp013 R1 v2 student SED",
                                        dir_mode="skip", quiet=False)
            print(f"Updated existing version: {DATASET_USER}/{DATASET_SLUG}")
        except Exception as e2:
            print(f"update also failed: {str(e2)[:200]}")
            print("Manual upload may be required.")

    print(f"\nFor blend NB: dataset_sources: ['{DATASET_USER}/{DATASET_SLUG}']")


## 完了

`maekeso/birdclef2026-exp013-r1-v2-student-sed` を blend NB の `dataset_sources` に指定して提出。

### 検証チェックリスト
- [ ] Cell 5 で pseudo_meta + Y_pseudo が正しく load された
- [ ] Cell 9 zip-streaming で sample 読み込み成功
- [ ] Cell 12 fold 0 の `non_s22_macro` が 0.85 以上 (R1 v1 の 0.67 を上回ること)
- [ ] Cell 13 ONNX verify で max|diff| < 1e-3
- [ ] Drive にミラー完了後、blend NB に dataset 指定して LB submission
